# 6 · Indexes — `loc` & `iloc`
*Intro to Python for Scientists & Public Health Professionals*

Selecting exactly the rows and columns you want is the most-used skill in pandas. Two accessors do almost all of it:

- **`loc`** selects by **label** (the index and column *names*)
- **`iloc`** selects by **integer position** (0-based, like NumPy)

Both take `[rows, columns]`.

### By the end of this notebook you can
- Select single values, lists, and slices with `loc` and `iloc`
- Filter rows with boolean conditions
- Avoid the classic traps: inclusive `loc` slices, `&`/`|` vs `and`/`or`, and integer indexes

### Agenda
1. `loc` vs `iloc`
2. Single values
3. Lists of labels / positions
4. Slices
5. Conditions (boolean selection)
6. When the index is integers

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

> **In plain language — `[rows, columns]`.** The comma inside the square brackets splits your request into two parts: what you want on the *left* (which rows) and what you want on the *right* (which columns). So `df.loc[rows, columns]` is really two selections happening at once. Leave one side as a bare `:` to mean "all of them" on that side.

In [ ]:
import pandas as pd

# One clinic's week: the index is the weekday (a *label*, not a position)
data = {
    "status":   ["Open", "Open", "Open", "Open", "Open", "Reduced", "Closed"],
    "visits":   [186, 175, 177, 176, 169, 78, 0],
    "wait_min": [31, 28, 30, 29, 27, 18, 0],
    "staff":    [12, 12, 11, 12, 10, 6, 1],
}
df = pd.DataFrame(data, index=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
df

## 2. Single values

`loc` uses labels; `iloc` uses positions. A bare `:` means "everything along that axis."  

> **In plain language — the bare `:` and "axis".** An *axis* is just a direction in the table: rows run down, columns run across. A bare colon `:` in a slot means "take everything along that direction." So `df.loc[:, "visits"]` reads as "*all* rows, but only the `visits` column," and `df.loc["Fri", :]` reads as "the `Fri` row, but *all* columns."

In [ ]:
# df.loc["Fri", "visits"]   # by label: Friday's visits
# df.iloc[4, 1]             # same cell by position: row 4, column 1

In [ ]:
df.loc[:, "visits"]    # all rows, one column  -> a Series

In [ ]:
df.loc["Fri", :]       # one row, all columns  -> a Series

> **In plain language — Series vs DataFrame.** A **DataFrame** is the whole table (many rows *and* many columns). A **Series** is a single strip of it — one column or one row pulled out on its own, keeping its labels. Whenever you narrow a selection down to a single column or a single row (as in the two cells above), pandas hands you back a Series rather than a full table.

> Reference: [Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html).

## 3. Lists of labels or positions

Pass a **list** inside the brackets to grab several rows or columns at once.

In [ ]:
df.loc[["Thu", "Fri"], ["visits", "wait_min"]]   # by label

In [ ]:
df.iloc[[3, 4], [1, 2]]                          # the same block by position

## 4. Slices

A key difference to internalize:

- **`loc` slices are *inclusive* of both endpoints** — `'Mon':'Thu'` includes Thursday.
- **`iloc` slices follow normal Python rules** — `0:4` stops *before* position 4.

> **In plain language — why the endpoints differ.** With `iloc` you're counting *positions*, and Python counting always stops just before the last number (`0:4` → 0, 1, 2, 3), exactly like normal list slicing. With `loc` you're naming *labels*, and it would be strange to name `'Thu'` as an endpoint and then not get Thursday — so `loc` includes both ends. Rule of thumb: position-slicing drops the last one, label-slicing keeps it.

In [ ]:
df.loc["Mon":"Thu", "status":"wait_min"]   # Mon..Thu AND status..wait_min, endpoints included

In [ ]:
df.iloc[0:4, 0:3]                          # rows 0,1,2,3 and cols 0,1,2 (stop excluded)

In [ ]:
df.loc["Mon":"Fri":2, :]                   # every other weekday: Mon, Wed, Fri

> **In plain language — the third `:` is a step.** A slice can take a third part: `start:stop:step`. The `step` says "how many to jump each time." So `"Mon":"Fri":2` starts at Mon and moves *2 at a time* → Mon, (skip Tue), Wed, (skip Thu), Fri. Leave the step off and it defaults to 1 (every one).

## 5. Conditions (boolean selection)

Filtering is the everyday use. Build a boolean Series and hand it to `loc`. This is the masking idea from NumPy, now keeping its labels.

> **In plain language — how boolean filtering works.** A comparison like `df["visits"] > 150` doesn't return a single True/False — it checks *every row* and gives back a whole column of True/False values (one per day). Handing that column to `loc` keeps only the rows marked True and drops the rest. That True/False column is the "boolean Series," and this keep-the-Trues idea is the heart of filtering.

In [ ]:
df.loc[df["visits"] > 150]            # only the busy days

For **multiple conditions**, wrap each in parentheses and join with `&` (and) or `|` (or). Python's `and`/`or` do **not** work element-wise on Series and will raise an error.

> **In plain language — why `and`/`or` break here.** Python's `and`/`or` expect a *single* True or False, but a filter condition is a whole column of them, so Python can't decide and raises an error. The symbols `&` and `|` are the versions that work value-by-value down the column. And each condition needs its own parentheses — `(df["visits"] > 150) & (df["status"] == "Open")` — otherwise `&` tries to grab the wrong pieces first.

In [ ]:
df.loc[(df["visits"] > 150) & (df["status"] == "Open"), ["visits", "wait_min"]]

To match against a *set* of allowed values, `.isin()` is cleaner than chaining several `|` conditions.

> **In plain language — what `.isin([...])` does.** It checks each value against a list of allowed options and returns True wherever the value is one of them. So `df["status"].isin(["Open", "Reduced"])` is a shorter way of writing "status is `Open` **or** status is `Reduced`" — handy when the list of allowed values gets long.

In [ ]:
df.loc[df["status"].isin(["Open", "Reduced"])]   # rows whose status is Open OR Reduced

### Exercise 1 — Select with labels and slices *(6 min)*

Using `df`: (a) get Wednesday's `wait_min` by label, and (b) return the `visits` and `staff` columns for Monday through Friday using a single `loc` slice.

In [ ]:
# Your work here


## 6. When the index is integers

This is the situation that confuses people. If the row index happens to be **integers**, `loc` and `iloc` can look the same — but `loc` still means *label* and `iloc` still means *position*. They diverge the moment the labels aren't `0,1,2,…`.

In [ ]:
df2 = df.reset_index(drop=True)   # replace weekday labels with a default 0..6 integer index
df2

In [ ]:
# df2.loc[2, "visits"]    # loc: the row *labeled* 2
# df2.iloc[2, 1]          # iloc: the row in *position* 2  (here, the same row)

They look identical here only because the labels equal the positions. After sorting or filtering, labels travel with their rows while positions renumber — so `loc[2]` and `iloc[2]` can point at different rows. Pick the accessor that matches what you mean.

> **In plain language — the integer-index trap, concretely.** Suppose you filter `df2` down to just the busy days and get back rows still *labeled* 0, 1, 4 (their original labels came along for the ride). Now `iloc[2]` gives you the **third row in the new order** (the one labeled 4), while `loc[2]` looks for the row **labeled** 2 — which got filtered out, so it errors. Labels stay stuck to their rows; positions always recount from 0 in whatever you're currently looking at. Reach for `loc` when you mean "the row *named* X" and `iloc` when you mean "the row *sitting in slot* X."

### Exercise 2 — Filter with a condition *(8 min)*

From `df`, return only the days where `visits` are above 150 **and** `status` equals `"Open"`, showing just the `visits` and `wait_min` columns.

In [ ]:
# Your work here


## Wrap-up

You can select by label (`loc`) and position (`iloc`), pull single values, lists, and slices, and filter rows with boolean conditions — while sidestepping the inclusive-slice and `&`/`|` traps.

**Next:** Inspecting & cleaning — turning a raw, messy load into analysis-ready data.